In [1]:
import os
#os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler
import random
#import torch
#import torch.nn.functional as F

#from deepod.models import USAD
#from deepod.models.time_series import TranAD
#from deepod.models.time_series import DeepIsolationForestTS
#from deepod.models.time_series import TcnED
#from deepod.models.time_series import DCdetector
#from deepod.models.time_series import AnomalyTransformer
#from deepod.models.time_series import DeepSVDDTS

from function import MLPRunner

def set_seed(seed=42):
    #os.environ['PYTHONHASHSEED'] = str(seed)
    #os.environ["OMP_NUM_THREADS"] = "1"
    #os.environ["MKL_NUM_THREADS"] = "1"
    #os.environ["OPENBLAS_NUM_THREADS"] = "1"
    #os.environ["NUMEXPR_NUM_THREADS"] = "1"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    #torch.use_deterministic_algorithms(True)
    #torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

#torch.backends.cudnn.deterministic = True

from sktime.classification.deep_learning.mlp import MLPClassifier

ModuleNotFoundError: No module named 'optuna'

In [ ]:
list2 = [['2026-01-19 13:31', '2026-01-19 14:44'], ['2026-01-20 09:02', '2026-01-20 11:29'], ['2026-01-23 08:54', '2026-01-23 10:15']]
list3 = [['2026-02-05 09:00', '2026-02-05 10:00'], ['2026-02-05 18:00', '2026-02-05 20:00'], ['2026-02-06 23:00', '2026-02-09 14:00'],
        ['2026-02-16 13:00', '2026-02-16 15:00']
]
list4 = [['2026-02-08 13:42', '2026-02-08 14:11'], ['2026-02-10 15:46', '2026-02-10 17:12'], ['2026-02-12 07:04', '2026-02-12 08:25']]
list6 = [['2025-12-13 17:23', '2025-12-13 17:30'], ['2025-12-13 18:41', '2025-12-13 18:44'], ['2025-12-19 12:19', '2025-12-19 12:40'],
         ['2025-12-22 18:59', '2025-12-22 19:00'], ['2026-01-24 22:01', '2026-01-24 22:02'], ['2026-01-31 22:09', '2026-01-31 23:57'],
         ['2026-02-04 16:47', '2026-02-04 16:54'], ['2026-02-04 17:14', '2026-02-04 17:28']
]
list7 = [['2026-03-03 09:15', '2026-03-03 09:30'], ['2026-03-03 13:10', '2026-03-03 13:20'], ['2026-03-04 13:00', '2026-03-04 13:20']]
list8 = [['2026-02-23 09:44', '2026-02-23 13:19'], ['2026-02-24 13:26', '2026-02-24 13:42'], ['2026-02-24 13 17:36', '2026-02-24 19:36']]

list_df = [1, 4, 6, 9, 11]

## sktime

In [ ]:
q = 0.95
train_model = True
use_miss = True
save_result = True
use_calibrate = False
scaler_cls = RobustScaler

In [ ]:
base_config = {
    "n_epochs": 80,
    "batch_size": 32,
    "activation": "relu",
    "verbose": False,
    "random_state": 42,
}

param_grid = {
    "n_epochs": (20, 200),
    "batch_size": [8, 16, 32, 64, 128],
    "activation": ["relu", "sigmoid"],
    "q": (0.85, 0.995),
    "alpha": (0.3, 3.0),
}

In [ ]:
runner = MLPRunner(score_mode="proba")

study = runner.tune_hyperparameters(
    num_data=1,
    param_grid=param_grid,
    base_config=base_config,
    model=MLPClassifier,
    n_trials=50
)

print(study.best_value)
print(study.best_params)
print(study.best_trial.user_attrs)

In [2]:
for num in [6]:
    print(f'Датасет {num}')
    config = {
        "n_epochs": 5,
        "verbose": True,
        "random_state": 42,
    }
    model = MLPClassifier
    model_name_save = 'MLPClassifier_base_'
    runner = MLPRunner(score_mode="predict")
    _ = runner.run(
            num_data=num,
            config=config,
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_deterministic=False
        )

Датасет 6


NameError: name 'MLPClassifier' is not defined

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epochs": 200,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = AnomalyTransformer
    model_name_save = 'Anomaly_transformer_epoch200_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=True
        )

## DCdetector

In [ ]:
q = 0.95
no_train = True
use_miss = True
save_result = True
use_calibrate = False
model = DCdetector
model_name_save = 'DCdetector_'
config = {
    "epochs": 80,
    #"batch_size": 64,
    "lr": 1e-3,
    "stride": 1,
    #'n_heads': 1,
    #'d_model': 128,
    #'e_layers': 3,
    #'threshold_': 1,
    #"epoch_steps": 20,
    "prt_steps": 20,
    "device": "cuda",
    "verbose": 1,
    "random_state": 42,
}

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    _ = train_func(
                model=model,
                config=config,
                num_data=num, 
                list_anomaly=globals()[f'list{num}'],
                model_name_save=model_name_save,
                model_name=f'{model_name_save[:-1]}',
                q = q,
                no_train=no_train,
                save_result=save_result,
                use_miss=use_miss,
                use_calibrate=use_calibrate
        )

## TcnED

In [ ]:
for num in range(2):
    print(f'Датасет {num}')
    config = {
        "epoch_steps": -1,
        "epochs": 5,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = TcnED
    model_name_save = 'exp'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=2,
            config=config,
            list_anomaly=globals()[f'list2'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=True
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epoch_steps": -1,
        "epochs": 80,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = TcnED
    model_name_save = 'TcnED_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epoch_steps": -1,
        "epochs": 200,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = TcnED
    model_name_save = 'TcnED_epoch200_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epochs": 80,
        "batch_size": 64,
        "lr": 1e-3,
        "stride": 1,
        'rep_dim': 32,
        'kernel_size': 3,
        'act': 'ReLU',
        "hidden_dims": 32,
        "bias": True,
        'dropout': 0.2,
        "epoch_steps": -1,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = TcnED
    model_name_save = 'TcnED_1_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epochs": 200,
        "batch_size": 64,
        "lr": 1e-3,
        "stride": 1,
        'rep_dim': 32,
        'kernel_size': 3,
        'act': 'ReLU',
        "hidden_dims": 32,
        "bias": True,
        'dropout': 0.2,
        "epoch_steps": -1,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = TcnED
    model_name_save = 'TcnED_epoch200_1_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls
        )

## DIF

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epoch_steps": -1,
        "epochs": 80,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = DeepIsolationForestTS
    model_name_save = 'DeepIFTS_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=False
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epoch_steps": -1,
        "epochs": 200,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = DeepIsolationForestTS
    model_name_save = 'DeepIFTS_epoch200_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=False
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epochs": 80,
        "batch_size": 256,
        "lr": 1e-3,
        "stride": 1,
        "hidden_dims": "64,32",
        "bias": False,
        "n_ensemble": 1,
        "n_estimators": 6,
        "max_samples": 256,
        "n_jobs": -1,
        "epoch_steps": -1,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = DeepIsolationForestTS
    model_name_save = 'DeepIFTS_1_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=False
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epochs": 200,
        "batch_size": 256,
        "lr": 1e-3,
        "stride": 1,
        "hidden_dims": "64,32",
        "bias": False,
        "n_ensemble": 1,
        "n_estimators": 6,
        "max_samples": 256,
        "n_jobs": -1,
        "epoch_steps": -1,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = DeepIsolationForestTS
    model_name_save = 'DeepIFTS_epoch200_1_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=False
        )

## TranAD

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epochs": 80,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = TranAD
    model_name_save = 'TranAD_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=False
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epochs": 200,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = TranAD
    model_name_save = 'TranAD_epoch200_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=False
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "stride": 1,
        "epochs": 80,
        "batch_size": 64,
        "lr": 5e-4,
        "epoch_steps": -1,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = TranAD
    model_name_save = 'TranAD_1_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=False
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "stride": 1,
        "epochs": 200,
        "batch_size": 64,
        "lr": 5e-4,
        "epoch_steps": -1,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = TranAD
    model_name_save = 'TranAD_epoch200_1_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=False
        )

## USAD

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epochs": 80,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = USAD
    model_name_save = 'USAD_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=False
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "epochs": 200,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = USAD
    model_name_save = 'USAD_epoch200_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=False
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "stride": 1,
        "hidden_dims": 64,
        'rep_dim': 32,
        "epochs": 80,
        "batch_size": 64,
        "lr": 1e-3,
        "es": 5,
        "train_val_pc": 0.1,
        "epoch_steps": -1,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = USAD
    model_name_save = 'USAD_1_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=False
        )

In [ ]:
for num in list_df:
    print(f'Датасет {num}')
    config = {
        "stride": 1,
        "hidden_dims": 64,
        'rep_dim': 32,
        "epochs": 200,
        "batch_size": 64,
        "lr": 1e-3,
        "es": 5,
        "train_val_pc": 0.1,
        "epoch_steps": -1,
        "prt_steps": 20,
        "device": "cuda",
        "verbose": 1,
        "random_state": 42,
    }
    model = USAD
    model_name_save = 'USAD_epoch200_1_'
    runner = DeepodRunner()
    _ = runner.run(
            num_data=num,
            config=config,
            list_anomaly=globals()[f'list{num}'],
            model_name_save=model_name_save,
            model_name=f'{model_name_save[:-1]}',
            q=q,
            model=model,
            save_result=save_result,
            train_model=train_model,
            use_calibrate=use_calibrate,
            scaler_cls=scaler_cls,
            use_deterministic=False
        )